# Update 2 — Offset-fair compensation improvement

**Revision note (v2.0):** The submitted manuscript quoted an 85 % reduction of the residual phase error relative to the Mathar model. That figure was dominated by a *static offset* of the Mathar model (1.166×10⁻⁶ in refractive index, 17.46 rad over the 4.2 m path). Any deployed system removes this constant offset with a single baseline calibration. This notebook quantifies the improvement **after** such a calibration, for both models equally.

**Metrics:**

$$I_{\rm amp} = 1 - \frac{\sigma_{\rm emp}}{\sigma_{\rm Mathar}},\qquad
  I_{\rm var} = 1 - \left(\frac{\sigma_{\rm emp}}{\sigma_{\rm Mathar}}\right)^2$$

with $\sigma_{\rm emp}$ the residual standard deviation of the empirical linear surrogate and $\sigma_{\rm Mathar}$ the residual standard deviation of the full nonlinear Mathar model **after subtracting its static offset**.

**Key numbers reproduced (cf. Supplemental Material, Sec. S3):**

| Quantity | Value |
|---|---|
| Static offset (Mathar − data) | 1.1659×10⁻⁶ |
| Offset phase over L = 4.2 m | 17.46 rad |
| σ_emp (linear surrogate) | 1.8367×10⁻⁷ |
| σ_Mathar (debiased, full model) | 1.9304×10⁻⁷ |
| I_amplitude | 4.86 % (95 % CI [3.92, 5.95]) |
| I_variance | 9.47 % (95 % CI [7.68, 11.5]) |

In [1]:
import pandas as pd
import numpy as np
import sys, os, time
sys.path.append('..')

from models.mathar.Mathar2007 import n as n_mathar_scalar

df = pd.read_csv('../../data/processed/full_data.csv')
df = df.sort_values('time').reset_index(drop=True)

T_C   = df['temperature'].values
H_pct = df['humidity'].values
P_hPa = df['pressure'].values
n_data = df['n_1762'].values

lam_um = 1.762
T_K = T_C + 273.15
P_Pa = P_hPa * 100.0

t0 = time.time()
n_mathar = np.array([n_mathar_scalar(lam_um, Tk, pp, hh)
                     for Tk, pp, hh in zip(T_K, P_Pa, H_pct)])
print(f"Mathar cloud generated in {time.time()-t0:.1f} s, N = {len(n_data)}")

Mathar cloud generated in 1.4 s, N = 145784


In [2]:
# Empirical linear surrogate residual
X = np.column_stack([np.ones(len(df)), T_C, H_pct, P_hPa])
beta_emp, *_ = np.linalg.lstsq(X, n_data, rcond=None)
resid_emp = n_data - X @ beta_emp
sigma_emp = np.std(resid_emp, ddof=0)
print(f"sigma_emp (linear surrogate)      = {sigma_emp:.10e}")

# Full nonlinear Mathar residual, debiased
resid_mathar_raw = n_data - n_mathar
offset_mathar = np.mean(resid_mathar_raw)
resid_mathar_db = resid_mathar_raw - offset_mathar
sigma_mathar = np.std(resid_mathar_db, ddof=0)

# phase conversions
lam = 1.762e-6
k_phase = 2*np.pi/lam                      # rad per meter per unit n
L_path = 4.2                               # m round trip
print(f"offset (Mathar - data)             = {offset_mathar:.10e}")
print(f"offset phase per meter             = {k_phase*offset_mathar:.3f} rad/m")
print(f"offset phase over L=4.2 m          = {k_phase*offset_mathar*L_path:.2f} rad")
print(f"sigma_Mathar (debiased, full model) = {sigma_mathar:.10e}")
print(f"sigma_emp  -> phase/m              = {k_phase*sigma_emp:.4f} rad/m")
print(f"sigma_Mathar -> phase/m            = {k_phase*sigma_mathar:.4f} rad/m")

sigma_emp (linear surrogate)      = 1.8366578763e-07
offset (Mathar - data)             = 1.1658573342e-06
offset phase per meter             = 4.157 rad/m
offset phase over L=4.2 m          = 17.46 rad
sigma_Mathar (debiased, full model) = 1.9303820448e-07
sigma_emp  -> phase/m              = 0.6549 rad/m
sigma_Mathar -> phase/m            = 0.6884 rad/m


In [3]:
# Point estimates
I_amp = 1 - sigma_emp/sigma_mathar
I_var = 1 - (sigma_emp/sigma_mathar)**2
print(f"I_amplitude = {I_amp:.4f}  ({100*I_amp:.2f} %)")
print(f"I_variance  = {I_var:.4f}  ({100*I_var:.2f} %)")

I_amplitude = 0.0486  (4.86 %)
I_variance  = 0.0947  (9.47 %)


In [4]:
# Moving-block bootstrap for confidence intervals (L = 156, B = 5000)
L = 156
B = 5000
N = len(df)
nb = N // L
rng = np.random.default_rng(42)

# Pre-aggregate per block
Xb_all = []
for b in range(nb):
    i0 = b*L
    Xb_all.append(np.column_stack([np.ones(L), T_C[i0:i0+L], H_pct[i0:i0+L], P_hPa[i0:i0+L]]))
Xb_all = np.array(Xb_all)                 # (nb, L, 4)

t0 = time.time()
I_amp_b, I_var_b = [], []
for i in range(B):
    blk = rng.integers(0, nb, size=nb)
    Xb  = Xb_all[blk].reshape(-1, 4)      # resampled rows (ordered by block)
    yd  = np.concatenate([n_data[b*L:(b+1)*L] for b in blk])
    ym  = np.concatenate([n_mathar[b*L:(b+1)*L] for b in blk])

    # empirical surrogate on resampled data
    beta, *_ = np.linalg.lstsq(Xb, yd, rcond=None)
    r_emp = yd - Xb @ beta
    s_emp = np.std(r_emp, ddof=0)

    # debiased full Mathar on same rows
    r_m = yd - ym
    s_m = np.std(r_m - r_m.mean(), ddof=0)

    I_amp_b.append(1 - s_emp/s_m)
    I_var_b.append(1 - (s_emp/s_m)**2)

I_amp_b = np.array(I_amp_b)
I_var_b = np.array(I_var_b)
print(f"Bootstrap completed in {time.time()-t0:.0f} s")

def ci95(a):
    return np.percentile(a, [2.5, 97.5])

print(f"I_amplitude = {I_amp:.4f}  95% CI [{ci95(I_amp_b)[0]:.4f}, {ci95(I_amp_b)[1]:.4f}]")
print(f"I_variance  = {I_var:.4f}  95% CI [{ci95(I_var_b)[0]:.4f}, {ci95(I_var_b)[1]:.4f}]")

Bootstrap completed in 26 s
I_amplitude = 0.0486  95% CI [0.0392, 0.0595]
I_variance  = 0.0947  95% CI [0.0768, 0.1154]
